# Imports

In [ ]:
import numpy as np
import pandas as pd
import time
import copy
import pickle
from pandas.tseries.holiday import USFederalHolidayCalendar

from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge, ElasticNet
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.ensemble import AdaBoostClassifier, AdaBoostRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier, MLPRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from catboost import CatBoostClassifier, CatBoostRegressor
from imblearn.under_sampling import RandomUnderSampler

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, RobustScaler
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    confusion_matrix, accuracy_score, balanced_accuracy_score,
    classification_report, top_k_accuracy_score, log_loss, roc_auc_score,
    mean_squared_error, r2_score, mean_absolute_error
)

**Basic global variables**

In [ ]:
seed = 42

# OPTION 1: Load raw data and do preprocessing

**Load in raw dataset (as "df")**

Otherwise, load in the cleaned dataset (see below).

Monthwise 2023 data can be downloaded from BTS: https://www.transtats.bts.gov/DL_SelectFields.aspx?gnoyr_VQ=FGJ&QO_fu146_anzr=b0-gvzr

Aggregated 2024 data can be downloaded from Kaggle: https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024

OPTION 1: Load sample dataset (2024 only)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2024_sample.csv')

OPTION 2: Load full (concatenated) 2023-24 dataset

*Warning: Use all available RAM on Colab or it will crash.*

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2023-24.csv')

OPTION 3: Load 2023 and 2024 data, and create full (concatenated) dataset

*Warning: Heavy RAM usage*

In [ ]:
## import 2023 data
jan23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - Jan.csv')
feb23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - Feb.csv')
mar23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - March.csv')
apr23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - April.csv')
may23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - May.csv')
jun23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - June.csv')
jul23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - July.csv')
aug23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - August.csv')
sep23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - Sept.csv')
oct23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - Oct.csv')
nov23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - Nov.csv')
dec23 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/T_ONTIME_REPORTING - Dec.csv')

list_of_dfs = [jan23,feb23,mar23,apr23,may23,jun23,jul23,aug23,sep23,oct23,nov23,dec23]

df_2023 = pd.concat(list_of_dfs, ignore_index=True)
df_2023.columns = df_2023.columns.str.lower()

## import 2024 data
df_2024 = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2024.csv')
df = pd.concat([df_2023,df_2024],ignore_index=True)

In [ ]:
# save full 2023-24 dataset
df.to_csv('flight_data_2023-24.csv', index=False)

**Preprocessing for raw data**

This isn't needed if you're loading the cleaned data directly (see below).

Drop rows with canceled or diverted flights:

In [ ]:
df = df[df['cancelled'].eq(0)]
df = df[df['diverted'].eq(0)]

Drop unneeded columns:

In [ ]:
keep = ['year', 'month', 'day_of_month', 'day_of_week', 'fl_date',
       'op_unique_carrier', 'origin', 'dest', 'crs_dep_time', 'dep_time',
       'crs_arr_time', 'arr_time', 'crs_elapsed_time', 'arr_delay']

df = df[keep]

Drop any row with missing values:

In [ ]:
df = df.dropna(axis=0, how="any")

Coerce text columns to strings:

In [ ]:
string_type = [
    'op_unique_carrier','origin', 'dest',
]
for c in string_type:
    df[c] = df[c].astype('string')

Coerce time columns to correct time datatype:

In [ ]:
df['crs_dep_time'] = pd.to_datetime(df['crs_dep_time'], format='%H%M', errors='coerce').dt.time
df['dep_time'] = pd.to_datetime(df['dep_time'], format='%H%M', errors='coerce').dt.time
df['crs_arr_time'] = pd.to_datetime(df['crs_arr_time'], format='%H%M', errors='coerce').dt.time
df['arr_time'] = pd.to_datetime(df['arr_time'], format='%H%M', errors='coerce').dt.time

Convert scheduled (crs_) arrival/departure times into hour and min columns:

In [ ]:
crs_cols = ['crs_dep_time', 'crs_arr_time']

for col in crs_cols:
    df[f"{col}_h"] = df[col].apply(lambda time_obj: time_obj.hour)
    df[f"{col}_m"] = df[col].apply(lambda time_obj: time_obj.minute)

Create season (fall/winter/etc) and holiday (Yes/No) columns from dates:

In [ ]:
df['fl_date'] = pd.to_datetime(df['fl_date'])

#seasons
season_map = {
    12:'winter', 1:'winter', 2:'winter',
    3:'spring', 4:'spring', 5:'spring',
    6:'summer', 7:'summer', 8:'summer',
    9:'fall', 10:'fall', 11:'fall'
}
df['season'] = pd.to_numeric(df['month'], errors='coerce').map(season_map).astype('category')

#holidays
if 'fl_date' in df.columns:
    df['fl_date'] = pd.to_datetime(df['fl_date'])

start = df['fl_date'].min() - pd.Timedelta(days=7)
end = df['fl_date'].max() + pd.Timedelta(days=7)
holidays = pd.DatetimeIndex(USFederalHolidayCalendar().holidays(start=start, end=end)).normalize()

d = df['fl_date'].dt.normalize().values.astype('datetime64[D]')
h = holidays.values.astype('datetime64[D]')
pos = np.searchsorted(h, d)
prev_idx = np.clip(pos - 1, 0, len(h) - 1)
next_idx = np.clip(pos, 0, len(h) - 1)
prev_diff = np.abs((d - h[prev_idx]).astype('timedelta64[D]').astype(int))
next_diff = np.abs((h[next_idx] - d).astype('timedelta64[D]').astype(int))
min_diff = np.minimum(prev_diff, next_diff)
df['holiday_flag'] = (min_diff <= 7).astype('int8')

More feature engineering:

*   **weekend_flag**: 1 for Sat/Sun flight, 0 otherwise
*   **dep_day_part**: 0 for night (12a-5:59a), 1 for morning (6-11:59a), 2 for afternoon (12-5:59p), 3 for evening (6-11:59p) - for scheduled departure time
*   **arr_day_part**: same, but for scheduled arrival time
*   **eastward_flag**: 1 for eastward flight, 0 for westward
*   **til_holiday**: number of days before/after nearest holiday (plus = days after holiday, minus = days before holiday)
*   **avg_delay_origin**: historical average delay for a given airport
*   **avg_delay_carrier**: historical average delay for a given carrier
*   **avg_delay_origin_hour**: historical average delay for a given airport at a given hour (e.g. ATL at 04:00)
*   **avg_delay_route**: historical average delay for a given route (origin->dest, as airport codes)


<!-- *   congestion_outbound: number of flights departing from the same airport within plus/minus 1 hour
*   congestion_inbound: number of flights landing at the same airport within plus/minus 1 hour
*   carrier_airport_pair: OP_UNIQUE_CARRIER-ORIGIN-->


In [ ]:
# flag for weekend vs weekday
df['weekend_flag'] = np.where(df['day_of_week'].isin([6, 7]), 1, 0)

# categorical variables for part of day
x = df['crs_dep_time_h']
dep_condition = [x < 6, x < 12, x < 18, x <= 23]
day_options = [0, 1, 2, 3]
df['dep_day_part'] = np.select(dep_condition, day_options, default=0)

y = df['crs_arr_time_h']
arr_condition = [y < 6, y < 12, y < 18, y <= 23]
df['arr_day_part'] = np.select(arr_condition, day_options, default=0)

In [ ]:
# flag for east/westward flight
airports = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/Visualization Files/airports.csv').set_index('iata')['longitude'].to_dict()

origin_long = df['origin'].map(airports)
dest_long = df['dest'].map(airports)

df['eastward_flag'] = (dest_long > origin_long).astype(int)

In [ ]:
# days before/after (-/+) nearest holiday

start = df['fl_date'].min() - pd.Timedelta(days=7)
end = df['fl_date'].max() + pd.Timedelta(days=7)
holidays = pd.DatetimeIndex(USFederalHolidayCalendar().holidays(start=start, end=end)).normalize()

d = df['fl_date'].dt.normalize().values.astype('datetime64[D]')
h = holidays.values.astype('datetime64[D]')
pos = np.searchsorted(h, d)
prev_idx = np.clip(pos - 1, 0, len(h) - 1)
next_idx = np.clip(pos, 0, len(h) - 1)

prev_diff = (d - h[prev_idx]).astype('timedelta64[D]').astype(int)
next_diff = (d - h[next_idx]).astype('timedelta64[D]').astype(int)

df['til_holiday'] = np.where(np.abs(prev_diff) <= np.abs(next_diff), prev_diff, next_diff)

In [ ]:
# avg_delay_origin: historical average delay for a given airport
origin_means = df.groupby('origin')['arr_delay'].mean().to_dict()
df['avg_delay_origin'] = df['origin'].map(origin_means)

# avg_delay_carrier: historical average delay for a given carrier
carrier_means = df.groupby('op_unique_carrier')['arr_delay'].mean().to_dict()
df['avg_delay_carrier'] = df['op_unique_carrier'].map(carrier_means)

# avg_delay_origin_hour: historical average delay for a given airport at a given hour (e.g. ATL at 04:00)
origin_hour_means = df.groupby(['origin', 'crs_dep_time_h'])['arr_delay'].mean().reset_index()
origin_hour_means = origin_hour_means.rename(columns={'arr_delay': 'avg_delay_origin_hour'})

df = df.merge(origin_hour_means, on=['origin', 'crs_dep_time_h'], how='left')
df['avg_delay_origin_hour'] = df['avg_delay_origin_hour'].fillna(df['avg_delay_origin'])

# avg_delay_route: historical average delay for a given route (origin->dest, as airport codes)
route_means = df.groupby(['origin', 'dest'])['arr_delay'].mean().reset_index()
route_means = route_means.rename(columns={'arr_delay': 'avg_delay_route'})

df = df.merge(route_means, on=['origin', 'dest'], how='left')
df['avg_delay_route'] = df['avg_delay_route'].fillna(df['avg_delay_origin'])

Create target variables:

*   total_delay: delay time in minutes (for regression)
*   delay_bucket: 4 categories of delay length (from None to 3hr+)

In [ ]:
df['total_delay'] = df['arr_delay']
df = df.drop(columns=["arr_delay"])

df['delay_bucket'] = pd.cut(
    df['total_delay'],
    bins=[-np.inf, 15, 60, 180, np.inf],
    labels=['No Delay', '16-60min delay', '1-3hr delay', '3hr+ delay'],
    right=True
)

In [ ]:
# Drop any rows with empty delay_bucket variable:
df = df.dropna(subset=['delay_bucket'])
df['delay_bucket'] = df['delay_bucket'].astype('category')

Drop unneeded columns after feature engineering:

In [ ]:
# Columns that leak delay amount: dep_time, arr_time, total_delay
# Unnecessary columns: fl_date, year, crs_dep_time_m, crs_arr_time_m
# Info captured in other features: origin, dest, op_unique_carrier, crs_dep_time, crs_arr_time

df = df.drop(columns=["fl_date", "year", "dep_time", "arr_time",
                      "crs_dep_time_m", "crs_arr_time_m", "origin", "dest",
                      "crs_dep_time", "crs_arr_time",
                      "total_delay"])

In [ ]:
print(df.columns)

Drop any rows with missing engineered features:

In [ ]:
df = df.dropna(axis=0, how="any")

In [ ]:
# Check for missing values - all values should be 0
print(df.isna().sum())

Create our own sample dataset for hyperparameter tuning:

(2% of the total size, matching the delay class distribution)

In [ ]:
target = 'delay_bucket'
_, df = train_test_split(df, test_size=0.02, random_state=42, shuffle=True, stratify=df[target])
df.reset_index(inplace=True)

**Save preprocessed data to csv**

OPTION 1: Save cleaned SAMPLE dataset (2024 only, ~10k rows)

In [ ]:
df.to_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2024_sample_cleaned.csv', index=False)

OPTION 2: Save our own cleaned SAMPLE dataset (2023-24, ~300k rows)

In [ ]:
df.to_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2024_samplev2_cleaned.csv', index=False)

OPTION 3: Save cleaned FULL dataset (2023-24, ~16m rows)

In [ ]:
df.to_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2023-24_cleaned.csv', index=False)

# OPTION 2: Load in preprocessed data from csv

**Load cleaned dataset (as "df")**

Use this if you didn't load and process data in the steps before this.

OPTION 1: Load cleaned SAMPLE dataset (2024 only, ~10k rows)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2024_sample_cleaned.csv')

OPTION 2: Load our own cleaned SAMPLE dataset (2023-24, ~300k rows)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2024_samplev2_cleaned.csv')

OPTION 3: Load cleaned FULL dataset (2023-24, ~16m rows)

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/DVA - Project Data/flight_data_2023-24_cleaned.csv')

Quick distribution check:

In [ ]:
# Optional: Check number of rows for each delay class
print(df.groupby('delay_bucket').size())

For full dataset:

Total number of observations: 13,672,473

```
1-3hr delay         766865   (5.60882%)
16-60min delay     1779509   (13.01527%)
3hr+ delay          185678   (1.35804%)
No Delay          10940421   (80.07187%)
```

For given sample dataset:

Total number of observations: 9,818
```
1-3hr delay         605   (6.16215%)
16-60min delay     1278   (13.01691%)
3hr+ delay          147   (1.49724%)
No Delay           7788   (79.32369%)
```

For our own sample dataset:

Total number of observations: 273,450

```
1-3hr delay        15337  (5.60870%)
16-60min delay     35590  (13.01518%)
3hr+ delay          3714  (1.35820%)
No Delay          218809  (80.01792%)
```

# Get data ready for model training

Split into train/test datasets:

In [ ]:
target = 'delay_bucket'
feature_cols = [c for c in df.columns if c != target]

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, shuffle=True, stratify=df[target])

X_train, y_train = train_df[feature_cols], train_df[target]
X_test, y_test = test_df[feature_cols], test_df[target]

In [ ]:
# Optional: Undersample the training dataset to make it balanced

rus = RandomUnderSampler(sampling_strategy='auto', random_state=42)
X_train, y_train = rus.fit_resample(X_train, y_train)

Set up preprocessing pipeline:

In [ ]:
# dtypes
num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]

# preprocess
preprocess = ColumnTransformer(
    transformers=[('num', StandardScaler(), num_cols),('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),])

# Test a range of classification models

Broad model testing/comparison - default models, without hyperparameter tuning

In [ ]:
dtree = DecisionTreeClassifier(
            random_state = seed,
            class_weight = "balanced"
        )
catboost = CatBoostClassifier(
            random_state = seed,
            verbose = 0,
            auto_class_weights = "Balanced"
        )
naivebayes = BernoulliNB()  # better suited for sparse data than GaussianNB
random_forest = RandomForestClassifier(
            random_state = seed,
            class_weight = "balanced",
            n_jobs = -1,
            max_depth = 5   # to avoid RAM overrun
        )
adaboost = AdaBoostClassifier(random_state = seed)
xgboost = XGBClassifier(
            random_state = seed,
            use_label_encoder = False,
            eval_metric = "mlogloss",
            verbosity = 0
        )
lightgbm = LGBMClassifier(
            random_state = seed,
            class_weight = "balanced",
            verbosity = -1
        )
neuralnet = MLPClassifier(
            random_state = seed,
            max_iter = 1000,
            verbose = False,
            early_stopping = True
        )
logreg = LogisticRegression(
            multi_class = "multinomial",
            max_iter = 2000,
            class_weight = "balanced",
            random_state = seed,
            verbose = 0
        )

all_models = {
    "dtree": dtree,
    "catboost": catboost,
    "naivebayes": naivebayes,
    "random_forest": random_forest,
    "adaboost": adaboost,
    "xgboost": xgboost,
    "lightgbm": lightgbm,
    "neuralnet": neuralnet,
    "logreg": logreg
    }
slow_models = {
    "knn": knn,
    "svm": svm
    }

In [ ]:
# Convert labels from strings to numbers (strings don't work for catboost)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

encoded_class_names = le.classes_
encoded_class_indices = le.transform(encoded_class_names)

In [ ]:
# RUNTIME: 65 mins
# Catboost 25 mins; adaboost 15m; neuralnet 6m; dtree, random_forest 5m;
# xgboost, lightgbm 3m; logreg, naivebayes <1m

results_final = {}

for model_name, model in all_models.items():
  print(f"Starting {model_name} model...")
  start = time.time()

  pipeline = Pipeline([("prep", preprocess), ("model", model)])

  pipeline.fit(X_train, y_train_encoded)

  # Calculate accuracy metrics

  y_test_pred = pipeline.predict(X_test)

  test_cm = confusion_matrix(y_test_encoded, y_test_pred, labels=encoded_class_indices)
  test_cm_df = pd.DataFrame(test_cm, index=[f"true_{c}" for c in encoded_class_names],
                                        columns=[f"pred_{c}" for c in encoded_class_names])

  accuracy = accuracy_score(y_test_encoded, y_test_pred)
  balanced_accuracy = balanced_accuracy_score(y_test_encoded, y_test_pred)
  report = classification_report(y_test_encoded, y_test_pred, labels=encoded_class_indices, target_names=encoded_class_names, digits=3)

  # Store results

  results_final[model_name] = {
      "accuracy": accuracy,
      "balanced_accuracy": balanced_accuracy,
      "confusion_matrix": test_cm_df,
      "report": report
  }

  print(f"Completed in {time.time()-start:.2f} secs\n")


In [ ]:
# Print comparison table for model results
print("\nSummary of results: ")
summary_df = pd.DataFrame.from_dict(results_final, orient='index')
print(summary_df[['accuracy', 'balanced_accuracy']].sort_values(by='balanced_accuracy', ascending=False))

Output from cell above (on full dataset):


```
Summary of results:
               accuracy  balanced_accuracy
catboost       0.504012           0.400213
lightgbm       0.500223           0.388966
random_forest  0.482480           0.345967
logreg         0.478668           0.341367
dtree          0.692872           0.309583
naivebayes     0.750744           0.281795
xgboost        0.800385           0.252557
neuralnet      0.800220           0.251020
adaboost       0.800178           0.250000
```

Output from cell above (on full dataset but with balanced training, imbalanced test dataset):


```
Summary of results:
               accuracy  balanced_accuracy
xgboost        0.497158           0.393424
catboost       0.496904           0.393142
lightgbm       0.498563           0.387949
neuralnet      0.467249           0.370818
adaboost       0.457827           0.346292
random_forest  0.485901           0.345813
logreg         0.478689           0.341294
naivebayes     0.500556           0.323513
dtree          0.336286           0.317722
```

# Hypertuning specific classification models

Hypertuning with 5-fold CV on the sample data (for time/RAM constraints):

In [ ]:
# Define each model and its tuning parameters
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)

classification_models = {
    "logreg": {
        "model": LogisticRegression(
            max_iter = 500,
            class_weight = "balanced",
            random_state = seed,
            verbose = 0
        ),
        "params": [
            {
                "model__solver": ["lbfgs"],
                "model__penalty": ["l2", None],
                # "model__C": np.logspace(-3, 1, 5)
                "model__C": np.logspace(-4, -2, 5)
            },
            {
                "model__solver": ["saga"],
                "model__penalty": ["elasticnet"],
                "model__C": np.logspace(-3, 1, 5),
                "model__l1_ratio": [0.1, 0.3, 0.5]
            }
        ]
    },
     "catboost": {
         "model": CatBoostClassifier(
            random_state=seed,
            verbose=0,
            loss_function="MultiClass",
      ),
      "params": [
          {
              "model__auto_class_weights": ["Balanced", "SqrtBalanced"],
              "model__iterations": [600, 900, 1000],  # default=1000
              "model__depth": [6, 8, 10],   # default=6
              "model__l2_leaf_reg": [3, 5, 7],   # default=3
          }
      ]
    },
    "lightgbm": {
        "model": LGBMClassifier(
            random_state=seed,
            class_weight="balanced",
            verbosity=-1
        ),
        "params": {
            "model__learning_rate": [0.01, 0.1],  # default: 0.1
            "model__n_estimators": [100, 200],  # default: 100
            "model__num_leaves": [31, 50]   # default: 31
        }
    },
    "random_forest": {
        "model": RandomForestClassifier(
            random_state=seed,
            class_weight="balanced",
            n_jobs=-1
        ),
        "params": {
            "model__n_estimators": [100, 200],  # default: 100
            "model__max_depth": [5, 10],  # default: None
            "model__min_samples_leaf": [1, 10]   # default: 1
        }
    }
}

In [ ]:
# Convert labels from strings to numbers (strings don't work for xgboost, catboost, lightgbm)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

encoded_class_names = le.classes_
encoded_class_indices = le.transform(encoded_class_names)

In [ ]:
# Fit/tune each model and store results
import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*'multi_class' was deprecated.*"
)

results = {}
classes = np.unique(y_test)

for model_name, model_info in classification_models.items():
    print(f"\nTUNING MODEL: {model_name}...")
    start = time.time()

    pipeline = Pipeline([("prep", preprocess), ("model", model_info["model"])])

    # Tune model across range of parameters
    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=model_info["params"],
        cv=cv,
        scoring="balanced_accuracy",
        n_jobs=-1,
        error_score="raise",
        verbose = 2
    )

    grid_search.fit(X_train, y_train_encoded)

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_

    # Calculate accuracy metrics

    y_test_pred = best_model.predict(X_test)

    test_cm = confusion_matrix(y_test_encoded, y_test_pred, labels=encoded_class_indices)
    test_cm_df = pd.DataFrame(test_cm, index=[f"true_{c}" for c in encoded_class_names],
                                          columns=[f"pred_{c}" for c in encoded_class_names])

    accuracy = accuracy_score(y_test_encoded, y_test_pred)
    balanced_accuracy = balanced_accuracy_score(y_test_encoded, y_test_pred)
    report = classification_report(y_test_encoded, y_test_pred, labels=encoded_class_indices, target_names=encoded_class_names, digits=3)

    # Store results

    results[model_name] = {
        "best_model": best_model,
        "best_params": best_params,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "confusion_matrix": test_cm_df,
        "report": report
    }

    print(f"Completed and stored in {time.time()-start:.2f} secs\n")

    print("Test confusion matrix:\n", test_cm_df)
    print("\nTest classification report:\n", report)

In [ ]:
for m, r in results.items():
    print(f"\nModel: {m}")
    print(f"Best parameters: {r["best_params"]}")
    print(f"Accuracy: {r["accuracy"]}")
    print(f"Balanced accuracy: {r["balanced_accuracy"]}")

Results:

```
Model: logreg
Best parameters: {'model__C': np.float64(0.001), 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Accuracy: 0.4610532089961602
Balanced accuracy: 0.34805408453946707

Model: catboost
Best parameters: {'model__auto_class_weights': 'Balanced', 'model__depth': 6, 'model__iterations': 1000, 'model__l2_leaf_reg': 3}
Accuracy: 0.5053122385024935
Balanced accuracy: 0.4022913548592395

Model: lightgbm
Best parameters: {'model__learning_rate': 0.1, 'model__n_estimators': 100, 'model__num_leaves': 31}
Accuracy: 0.5068933991588956
Balanced accuracy: 0.3690354518296612

Model: random_forest
Best parameters: {'model__max_depth': 10, 'model__min_samples_leaf': 10, 'model__n_estimators': 200}
Accuracy: 0.5235509233863594
Balanced accuracy: 0.35348578880466486
```

In [ ]:
# Print comparison table for model results
print("\nSummary of results: ")
summary_df = pd.DataFrame.from_dict(results, orient='index')
print(summary_df[['accuracy', 'balanced_accuracy']].sort_values(by='balanced_accuracy', ascending=False))

Results (on sample data):

```
Summary of results:
               accuracy  balanced_accuracy
catboost       0.505312           0.402291
logreg         0.461053           0.348054
lightgbm       0.506893           0.369035
random_forest  0.523551           0.353486
```

**Calculate accuracy stats for best parameters on the full dataset:**

In [ ]:
# Best parameters from above
best_lr_params = {'C': 0.001, 'penalty': 'l2', 'solver': 'lbfgs'}
best_cb_params = {'auto_class_weights': 'Balanced', 'depth': 6, 'iterations': 1000}
best_rf_params = {'max_depth': 10, 'min_samples_leaf': 10, 'n_estimators': 200}
best_lg_params = {'learning_rate': 0.1, 'n_estimators': 100, 'num_leaves': 31}

classification_models = {
    "lightgbm": {
        "model": LGBMClassifier(
            random_state=seed,
            class_weight="balanced",
            verbosity=-1
        ),
        "params": {}
    },
    "random_forest": {
        "model": RandomForestClassifier(
            random_state=seed,
            class_weight="balanced",
            n_jobs=-1
        ),
        "params": {}
    },
    "logreg": {
        "model": LogisticRegression(
            max_iter=500,
            class_weight="balanced",
            random_state=seed,
            verbose=0
        ),
        "params": {}
    },
    "catboost": {
        "model": CatBoostClassifier(
            random_state=seed,
            verbose=0
        ),
        "params": {}
    }
}

In [ ]:
# Convert labels from strings to numbers (strings don't work for xgboost, catboost, lightgbm)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

encoded_class_names = le.classes_
encoded_class_indices = le.transform(encoded_class_names)

In [ ]:
results_fixed = {}
for name, info in classification_models.items():
    print(f"\nFITTING FIXED MODEL: {name}")
    start = time.time()

    pipe = Pipeline([("prep", preprocess), ("model", info["model"])])

    if name == "logreg":
      pipe.named_steps["model"].set_params(**best_lr_params)
    elif name == "catboost":
        pipe.named_steps["model"].set_params(**best_cb_params)
    elif name == "random_forest":
        pipe.named_steps["model"].set_params(**best_rf_params)
    elif name == "lightgbm":
        pipe.named_steps["model"].set_params(**best_lg_params)

    pipe.fit(X_train, y_train_encoded)

    y_pred = pipe.predict(X_test)
    acc  = accuracy_score(y_test_encoded, y_pred)
    bacc = balanced_accuracy_score(y_test_encoded, y_pred)

    cm = confusion_matrix(y_test_encoded, y_pred, labels=encoded_class_indices)
    cm_df = pd.DataFrame(cm,
        index=[f"true_{c}" for c in encoded_class_names],
        columns=[f"pred_{c}" for c in encoded_class_names])

    rpt = classification_report(
        y_test_encoded, y_pred, labels=encoded_class_indices,
        target_names=encoded_class_names, digits=3
    )

    results_fixed[name] = {
        "best_params": (best_lr_params if name=="logreg" else best_cb_params if name=="catboost" else best_rf_params if name=="random_forest" else best_lg_params),
        "accuracy": acc,
        "balanced_accuracy": bacc,
        "confusion_matrix": cm_df,
        "report": rpt,
        "best_model": pipe
    }

    print(f"Done in {time.time()-start:.2f}s — test acc={acc:.3f} | bal_acc={bacc:.3f}")
    print("Test confusion matrix:\n", cm_df)
    print("\nTest classification report:\n", rpt)

In [ ]:
# Print comparison table for model results
print("\nSummary of results: ")
summary_df = pd.DataFrame.from_dict(results_fixed, orient='index')
print(summary_df[['accuracy', 'balanced_accuracy']].sort_values(by='balanced_accuracy', ascending=False))

Results:

```
Summary of results:
               accuracy  balanced_accuracy
catboost       0.505312           0.402291
lightgbm       0.500202           0.389796
random_forest  0.486943           0.368911
logreg         0.455638           0.353998
```

# Set up final Catboost model for visualization

Set up final Catboost model, using tuned parameters from above, and save for use in the visualization:

In [ ]:
best_cb_params = {'auto_class_weights': 'Balanced', 'depth': 6, 'iterations': 1000}

cb_model = CatBoostClassifier(
              random_state=seed,
              verbose=0
          )

In [ ]:
# Convert labels from strings to numbers (strings don't work for xgboost, catboost, lightgbm)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

encoded_class_names = le.classes_
encoded_class_indices = le.transform(encoded_class_names)

In [ ]:
print(f"\nFITTING FINAL CATBOOST MODEL...")
start = time.time()

pipe = Pipeline([("prep", preprocess), ("model", cb_model)])
pipe.named_steps["model"].set_params(**best_cb_params)
pipe.fit(X_train, y_train_encoded)

y_pred = pipe.predict(X_test)

acc  = accuracy_score(y_test_encoded, y_pred)
bacc = balanced_accuracy_score(y_test_encoded, y_pred)

cm = confusion_matrix(y_test_encoded, y_pred, labels=encoded_class_indices)
cm_df = pd.DataFrame(cm,
    index=[f"true_{c}" for c in encoded_class_names],
    columns=[f"pred_{c}" for c in encoded_class_names])

rpt = classification_report(
    y_test_encoded, y_pred, labels=encoded_class_indices,
    target_names=encoded_class_names, digits=3
)

pickle.dump(pipe, open("catboost_pipe.pkl", "wb"))

print(f"Done in {time.time()-start:.2f}s — test acc={acc:.3f} | bal_acc={bacc:.3f}")
print("Test confusion matrix:\n", cm_df)
print("\nTest classification report:\n", rpt)

Results:


```
FITTING FINAL CATBOOST MODEL...
Done in 1489.37s — test acc=0.505 | bal_acc=0.402
Test confusion matrix:
                      pred_1-3hr delay  pred_16-60min delay  pred_3hr+ delay  \
true_1-3hr delay                43809                29683            46675   
true_16-60min delay             73761               100251            75084   
true_3hr+ delay                  7551                 3880            17989   
true_No Delay                  234845               381736           351780   

                     pred_No Delay  
true_1-3hr delay             33206  
true_16-60min delay         106806  
true_3hr+ delay               7716  
true_No Delay              1219723  

Test classification report:
                 precision    recall  f1-score   support

   1-3hr delay      0.122     0.286     0.171    153373
16-60min delay      0.194     0.282     0.230    355902
    3hr+ delay      0.037     0.484     0.068     37136
      No Delay      0.892     0.557     0.686   2188084

      accuracy                          0.505   2734495
     macro avg      0.311     0.402     0.289   2734495
  weighted avg      0.746     0.505     0.589   2734495
```



# Risk index

In [ ]:
probs = pipeline.predict_proba(X_test)
classes = pipeline.named_steps['model'].classes_
weights = {
    3 : 0, # 3 -> No Delay
    1 : 10, # 1 -> 16-60min delay
    0: 40, # 0 -> 1-3hr delay
    2 : 100 # 2 -> 3hr+ delay
}

W = np.array([weights[c] for c in classes], dtype=float)
risk_raw = probs @ W
risk_0_100 = (risk_raw / W.max()) * 100
scored = test_df.copy()
scored['risk_index'] = risk_0_100.round(1)